In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("Working dir:", os.getcwd())

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

# Adjust path below to where your step7_pca.csv is located in your Drive
RAW_PATH = "/content/drive/MyDrive/Colab Notebooks/data/processed/Dataset7.csv"
if not os.path.exists(RAW_PATH):
    # fallback: try current folder
    RAW_PATH = "Dataset7.csv"

df = pd.read_csv(RAW_PATH)
print("Loaded:", RAW_PATH, "shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


In [ ]:
# Cell 2: Prepare features and target (shared)
from sklearn.model_selection import train_test_split

# Ensure target exists
assert 'selling_price' in df.columns, "selling_price not in dataset"

X = df.drop(columns=['selling_price'])
y = df['selling_price']

# If any NaNs remain, fill with median for modeling convenience
X = X.fillna(X.median())

# Train-test split (same for all members to ensure fair comparison)
RANDOM_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

trees = [50, 100, 200]
rf_results = []
for n in trees:
    rf = RandomForestRegressor(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    yp = rf.predict(X_test)
    rf_results.append((n, r2_score(y_test, yp), mean_absolute_error(y_test, yp)))

for n, r2v, mae in rf_results:
    print(f"n_trees={n} -> R2: {r2v:.3f}, MAE: {mae:.0f}")

# Fit final RF (highest trees) and show importances
best_n = max(rf_results, key=lambda t: t[1])[0]
rf_best = RandomForestRegressor(n_estimators=best_n, random_state=RANDOM_SEED, n_jobs=-1)
rf_best.fit(X_train, y_train)
y_pred_rf = rf_best.predict(X_test)

# Feature importances
importances = pd.Series(rf_best.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(8,5))
importances.plot.barh()
plt.title(f"Random Forest Feature Importances (n={best_n})")
plt.show()

print("Random Forest reduces variance and often gives better accuracy than single trees. "
      "Feature importances show which input components most influence price.")
